# QQQ负bar时SMH相对抗跌：固定五会话信息门

来源公式、四头Ridge和全部门在2021/22经济值前冻结。匹配控制含原39、开盘/当前账户、普通SOXL/SOXX分钟价量、旧自身回撤、旧AMD/NVDA/SMH 09:45同涨、QQQ当前5m收益及无条件SMH−QQQ价差。候选仅增加QQQ负bar×SMH相对价差。2021/22是已复用的开发评估，不称独立验证。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
warnings.filterwarnings('ignore',message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
from src.core.recorder_run import publish_evidence_manifest
from src.analysis.stage1_fixed_share_drawdown_information import prepare_feature_panel,fit_information_screen
from src.analysis.stage1_peer_downshock_information import attach_peer_downshock
root=Path.cwd();card=root/'research/stage1_v5_peer_downshock_information_20261030';plan_bytes=(card/'plan.json').read_bytes();plan=json.loads(plan_bytes);sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan['core_provider_uri']/'qlib_store_manifest.json')==plan['core_provider_manifest_sha256'] and sha(root/plan['peer_provider_uri']/'qlib_store_manifest.json')==plan['peer_provider_manifest_sha256']
assert sha(root/plan['peer_source_card']/'summary.json')==plan['peer_source_summary_sha256'] and sha(root/plan['peer_source_card']/'audit.json')==plan['peer_source_audit_sha256']
assert json.loads((root/plan['peer_parent_card']/'nas_binding.json').read_text())['evidence']['sha256']==plan['peer_parent_nas_archive_sha256']
for role in ('label_source','parent','frame'):
 s=plan[role];source=root/s['card'];assert sha(source/'summary.json')==s['summary_sha256'] and sha(source/'verification_manifest.json')==s['manifest_sha256'] and json.loads((source/'nas_binding.json').read_text())['evidence']['sha256']==s['nas_archive_sha256']
qlib.init(provider_uri=str(root/plan['peer_provider_uri']),region='us',kernels=1)
peer=D.features(['QQQ','SMH','NVDA','AMD'],['$open','$close','$price_valid'],start_time='2018-10-15 09:35:00',end_time='2022-12-22 16:00:00',freq='5min')
print('Qlib peer5m rows',len(peer))

[51130:MainThread](2026-10-30 17:16:30,124) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[51130:MainThread](2026-10-30 17:16:30,126) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[51130:MainThread](2026-10-30 17:16:30,126) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/peer_5min')}


Qlib peer5m rows 328176


In [2]:
qlib.init(provider_uri=str(root/plan['core_provider_uri']),region='us',kernels=1)
def read(role,name):
 s=plan[role];v,_=load_verified_recorder_objects(root/s['card'],expected_recorder_id=s['recorder_id'],artifacts=[name],expected_manifest_sha256=s['manifest_sha256']);return v[name]
labels=read('label_source','analysis/labels');book=read('parent','analysis/native/dated_f696_0');frame=read('frame','inputs/covered_frame').reset_index(drop=True)
core=D.features(['SOXL','SOXX'],['$close','$raw_volume','$price_valid'],start_time='2018-10-15 09:31:00',end_time='2022-12-22 16:00:00',freq='1min')
core_plan=dict(plan)
core_plan['state_features']=[x for x in plan['state_features'] if x!='own_drawdown_change']
core_plan['ordinary_minute_features']=plan['ordinary_minute_features'][:6]
core_plan['candidate_only_feature']='own_drawdown_change'
panel,core_source=prepare_feature_panel(labels,frame,book,core,core_plan)
panel,peer_source=attach_peer_downshock(panel,peer)
assert len(panel)==len(labels)==385932
source_eval=panel.loc[panel.trade_date.dt.year.isin([2021,2022]) & panel.trade_date.le(pd.Timestamp(plan['evaluation_end']))]
annual={x['year']:x for x in peer_source['annual']};gate=plan['source_gate']
source_pass=bool(annual[2021]['days']>=gate['minimum_2021_mapped_days'] and source_eval.loc[source_eval.trade_date.dt.year.eq(2022),'trade_date'].nunique()>=gate['minimum_2022_mapped_mature_days'] and peer_source['maximum_peer_age_minutes']<=gate['maximum_peer_bar_age_minutes'] and panel[['qqq_current_5m_return','smh_qqq_current_5m_gap','peer_breadth_0945','peer_downshock_resilience']].notna().all().all())
source_gate={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'core_source':core_source,'peer_source':peer_source,'source_pass':source_pass,'reused_2021_2022_economic_values_accessed_before_fit':False}
(card/'source_gate.json').write_text(json.dumps(source_gate,ensure_ascii=False,indent=2)+'\n')
if not source_pass:raise ValueError('blocked: held peer source clock/coverage failed before economic fit')
print(json.dumps({'source_pass':source_pass,'annual':peer_source['annual'],'maximum_peer_age_minutes':peer_source['maximum_peer_age_minutes']},ensure_ascii=False))

[51130:MainThread](2026-10-30 17:16:30,450) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[51130:MainThread](2026-10-30 17:16:30,451) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[51130:MainThread](2026-10-30 17:16:30,451) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{"source_pass": true, "annual": [{"year": 2018, "rows": 19091, "days": 53, "max_peer_age_minutes": 4.0}, {"year": 2019, "rows": 91944, "days": 252, "max_peer_age_minutes": 4.0}, {"year": 2020, "rows": 92491, "days": 253, "max_peer_age_minutes": 4.0}, {"year": 2021, "rows": 92304, "days": 252, "max_peer_age_minutes": 4.0}, {"year": 2022, "rows": 90102, "days": 246, "max_peer_age_minutes": 4.0}], "maximum_peer_age_minutes": 4.0}


In [3]:
screen=fit_information_screen(panel,labels,plan)
metrics=[{'model':model,'target':target,'year':int(year),**value} for (model,target,year),value in screen['metrics'].items()];m={(x['model'],x['target'],x['year']):x for x in metrics}
annual=screen['annual_actions'];g={}
g['control_beats_training_mean']=all(m[('control',target,y)]['day_mse']<m[('control',target,y)]['train_mean_mse'] for target in plan['targets'] for y in plan['evaluation_years'])
g['candidate_beats_control_mse']=all(m[('candidate',target,y)]['day_mse']<m[('control',target,y)]['day_mse'] for target in plan['targets'] for y in plan['evaluation_years'])
features=[*plan['base39_features'],*plan['state_features'],*plan['ordinary_minute_features'],plan['candidate_only_feature']]
model=screen['models'][('candidate','soxx_value')];names=list(model.named_steps['simpleimputer'].get_feature_names_out(features));where=names.index(plan['candidate_only_feature']);coefficient=float(model.named_steps['ridge'].coef_[where]);g['soxx_added_coefficient_positive']=coefficient>0
g['candidate_control_funded_action_change']=sum(x['candidate_control_action_changed_days'] for x in annual)>=plan['gates']['candidate_control_funded_action_changed_days_min']
g['same_model_zero_action_change']=sum(x['same_model_zero_action_changed_days'] for x in annual)>=plan['gates']['same_model_zero_action_changed_days_min']
g['candidate_control_value_positive_each_year']=all(x['candidate_minus_control_value_sum']>0 for x in annual)
g['candidate_hold_value_positive_each_year']=all(x['candidate_value_sum']>0 for x in annual)
g['candidate_control_block_lower_positive']=screen['block_lower']>0
g['candidate_tail_damage_not_worse_each_year']=all(x['candidate_minus_control_tail_damage_sum']<=0 for x in annual)
passed=all(g.values())
daily=screen['daily_actions'].copy();daily['quarter']=daily.trade_date.dt.to_period('Q').astype(str)
quarter=[]
for q,part in daily.groupby('quarter',sort=True):quarter.append({'quarter':q,'days':len(part),'candidate_value_sum':float(part.value_candidate.sum()),'control_value_sum':float(part.value_control.sum()),'candidate_minus_control_value_sum':float(part.candidate_minus_control.sum()),'candidate_minus_control_tail_sum':float(part.tail_candidate_minus_control.sum())})
pred=pd.DataFrame({'trade_date':pd.to_datetime(screen['evaluation_day_index']),'control_soxl':screen['predictions'][('control','soxl_value')],'control_soxx':screen['predictions'][('control','soxx_value')],'candidate_soxl':screen['predictions'][('candidate','soxl_value')],'candidate_soxx':screen['predictions'][('candidate','soxx_value')],'candidate_zero_soxl':screen['zero_predictions']['soxl_value'],'candidate_zero_soxx':screen['zero_predictions']['soxx_value']})
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'peer_code_sha256':sha(root/'src/analysis/stage1_peer_downshock_information.py'),'source_gate_pass':source_pass,'train_rows':screen['train_rows'],'train_days':screen['train_days'],'evaluation_rows':screen['eval_rows'],'evaluation_days':screen['eval_days'],'metrics':metrics,'candidate_soxx_added_coefficient':coefficient,'relative_soxl_minus_soxx_added_coefficient':screen['relative_feature_coefficient'],'prediction_changed_rows':screen['prediction_changed_rows'],'same_model_zero_score_changed_rows':screen['same_model_zero_score_changed_rows'],'annual_actions':annual,'quarter_actions':quarter,'candidate_control_block_lower':screen['block_lower'],'gates':g,'information_gate_pass':passed,'new_model_fits':4,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'conditional_peer_downshock_information_pass_needs_continuous_target' if passed else 'not_promoted_fixed_conditional_peer_downshock_information_failed'}
print(json.dumps(audit,ensure_ascii=False,indent=2))

[51130:MainThread](2026-10-30 17:16:33,604) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[51130:MainThread](2026-10-30 17:16:33,604) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[51130:MainThread](2026-10-30 17:16:33,604) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Init data Done


{
  "experiment": "stage1_v5_peer_downshock_information_20261030",
  "plan_sha256": "714e20047eed5c1f976d7cf5f306da9303ac2fb31083778058afc437de7205a2",
  "peer_code_sha256": "d01782053db1b9a93ab6fd76f087cf4efb5fa9248ba8a4cbbf7860842207389a",
  "source_gate_pass": true,
  "train_rows": 200403,
  "train_days": 549,
  "evaluation_rows": 180938,
  "evaluation_days": 494,
  "metrics": [
    {
      "model": "control",
      "target": "soxl_value",
      "year": 2021,
      "days": 252,
      "day_mse": 2.111215680596939e-05,
      "train_mean_mse": 1.2022830468401984e-05
    },
    {
      "model": "control",
      "target": "soxl_value",
      "year": 2022,
      "days": 242,
      "day_mse": 0.00018242843727108875,
      "train_mean_mse": 3.0208267843215525e-05
    },
    {
      "model": "candidate",
      "target": "soxl_value",
      "year": 2021,
      "days": 252,
      "day_mse": 2.11094294808157e-05,
      "train_mean_mse": 1.2022830468401984e-05
    },
    {
      "model": "candid

In [4]:
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_gate=source_gate)
 rec.save_objects(artifact_path='sources',peer_join_code=(root/'src/analysis/stage1_peer_downshock_information.py').read_text())
 rec.save_objects(artifact_path='models',control_soxl=screen['models'][('control','soxl_value')],control_soxx=screen['models'][('control','soxx_value')],candidate_soxl=screen['models'][('candidate','soxl_value')],candidate_soxx=screen['models'][('candidate','soxx_value')])
 rec.save_objects(artifact_path='analysis',predictions=pred,daily_actions=daily,annual_actions=annual,quarter_actions=quarter,metrics=metrics,audit=audit)
 rec.log_metrics(train_days=float(screen['train_days']),evaluation_days=float(screen['eval_days']),action_changed_days=float(sum(x['candidate_control_action_changed_days'] for x in annual)),candidate_control_block_lower=float(screen['block_lower']),information_gate_pass=float(passed))
assert rec.status=='FINISHED'
summary={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'decision':audit['decision'],'historical_research_passed':False,'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/str(rec.experiment_id)/rec.id),'source_gate':source_gate,'train_rows':screen['train_rows'],'train_days':screen['train_days'],'evaluation_rows':screen['eval_rows'],'evaluation_days':screen['eval_days'],'gates':g,'annual_actions':annual,'quarter_actions':quarter,'candidate_control_block_lower':screen['block_lower'],'new_model_fits':4,'new_candidate_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','sources/*','models/*','analysis/*']},'limitations':['2021/22 economic values were reused development feedback from a prior failed candidate, not independent validation.','This is local five-session first-action information, not a complete SOXX candidate account or continuous allocator.','Historical first 5m receipt/finality, broker fills and dividend cash remain unqualified.']}
(card/'workflow.yaml').write_text('experiment: '+card.name+'\nnotebook: '+card.name+'.ipynb\nrecorder_uri: '+str(tracking)+'\nartifact_policy: slim_card_recorder_first\n')
publish_evidence_manifest(card,summary);(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'recorder_id':rec.id,'information_gate_pass':passed,'train_days':screen['train_days'],'eval_days':screen['eval_days']},ensure_ascii=False))

[51130:MainThread](2026-10-30 17:16:36,674) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_v5_peer_downshock_information_20261030.


[51130:MainThread](2026-10-30 17:16:36,676) INFO - qlib.workflow - [exp.py:258] - Experiment 739162178795115186 starts running ...


[51130:MainThread](2026-10-30 17:16:36,748) INFO - qlib.workflow - [recorder.py:345] - Recorder bfd3400477d24a838181e97e01c6ebba starts running under Experiment 739162178795115186 ...


[51130:MainThread](2026-10-30 17:16:37,525) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{"recorder_id": "bfd3400477d24a838181e97e01c6ebba", "information_gate_pass": false, "train_days": 549, "eval_days": 494}


## 中文结论

SOXX历史Goal未通过；固定QQQ负bar×SMH相对抗跌信息版关闭。Qlib peer5m与core1m在2018–2022全部385932事前意向点时覆盖，最长同行bar老4分钟；2018–2020成熟200403行/549日训练四头，复用2021/22 180938行/494日评估。新字段SOXX系数+1.44e−6、分数有变化，但候选/控制实际首行动仅差4/1日，同模型归零仅差7/1日；SOXX头MSE两年略差于控制，所有头差于训练均值常数，10会话块下界−1.03e−6，2022候选对HOLD局部和−0.109755。2022Q2 62日全09:48行动、61日仍买SOXX，首分预测+91.748bp而实际−20.859bp；2021Q3微增益、2021Q2损伤，不能挑季度。Qlib Recorder `bfd3400477d24a838181e97e01c6ebba`及NAS七件对象远端读回通过。0连续目标、0候选账户，失败版不调参。